# 投机解码：一次 Target Forward 能不能确认多个 Token？

> 上一章通过量化降低了每一步 Decode 的权重与缓存开销，但自回归生成的基本限制没有改变：
>
> ```text
> 生成 token 1
> → 才知道 token 2 的输入
> → 再生成 token 2
> → 才知道 token 3 的输入
> ```
>
> 本章只回答一个问题：**能不能一次昂贵的 Target Model Forward，确认多个 Token？**
>
> 这就是 **Speculative Decoding / Speculative Sampling**。
>
> 我们按完整流程走：
>
> 1. Draft Model 先快速“猜”多个 Token。
> 2. Target Model 一次并行验证这些位置。
> 3. 根据接受概率保留尽可能长的前缀。
> 4. 如果拒绝，用修正分布重新采样。
> 5. 最后验证：加速以后，输出分布是否仍然和 Target-only 一致。

先从串行瓶颈开始。

## 1. 自回归生成的串行依赖

假设 Target Model 每次 Forward 需要 40 ms。

生成 4 个 Token：

```text
40ms → token 1
40ms → token 2
40ms → token 3
40ms → token 4
```

仅 Target Forward 就是 160 ms。

即使 Token 很“好猜”，Target Model 也必须一步一步确认。

投机解码的直觉是：

> **让便宜的小模型先把后面几步猜出来，再让大模型一次性检查。**

In [ ]:
target_step_ms = 40
tokens = 4
print("普通自回归理论 Target 时间:", target_step_ms * tokens, "ms")

draft_step_ms = 5
draft_tokens = 4
verify_ms = 45
print("假设 Draft 先猜 4 个，再一次 Target 验证:")
print("Draft 时间:", draft_step_ms * draft_tokens, "ms")
print("Target 验证:", verify_ms, "ms")
print("如果全部接受，总时间:", draft_step_ms * draft_tokens + verify_ms, "ms")

注意，这只是建立直觉。

真实 speedup 取决于：

```text
Draft Model 有多快
×
每轮猜多少 Token
×
Target 一次验证的成本
×
最终接受率
```

如果 Draft 很慢，或者猜得总是错，投机解码可能反而没有收益。

## 2. Draft → Verify：完整流程先走一遍

假设当前上下文是：

```text
今天 天气
```

Draft Model 连续生成：

```text
真 好 啊 !
```

Target Model 不需要再一个 Token 一个 Token 地跑。

它可以把：

```text
今天 天气 真 好 啊 !
```

一次送进去，同时得到多个位置的概率。

然后从左到右检查 Draft 的每个候选。

```text
Draft:   真    好    啊    !
Target:  ✓     ✓     ✗
```

一旦第 3 个位置拒绝：

- 前两个接受。
- 第 3 个位置由 Target 的修正分布重新采样。
- 第 4 个候选及之后全部丢弃。
- 开启下一轮 Speculation。

## 3. 为什么不是“Draft 猜中 Target Top-1 就接受”？

这是投机解码最容易讲错的地方。

如果我们只是做：

```text
Draft token == Target argmax
→ accept
否则 reject
```

对于 Greedy Decoding 可以构造类似思想，但完整的 **Speculative Sampling** 要求更强：

> 最终采样分布必须和“直接从 Target Model 采样”完全一致。

经典接受概率是：

\[
a(x)=\min\left(1,\frac{p(x)}{q(x)}\right)
\]

- `p(x)`：Target 对 Draft Token 的概率
- `q(x)`：Draft 对这个 Token 的概率

如果 Target 比 Draft 更看好这个 Token，直接接受。

如果 Draft 过度自信，就按比例随机接受。

In [ ]:
import random

draft_tokens = [15, 23, 8, 42]
draft_probs  = [0.8, 0.7, 0.6, 0.5]
target_probs = [0.9, 0.8, 0.2, 0.1]

for tok, q, p in zip(draft_tokens, draft_probs, target_probs):
    accept_prob = min(1.0, p / q)
    print(f"token={tok:2d}  draft={q:.2f} target={p:.2f} -> accept_prob={accept_prob:.2%}")

这里要特别区分两个概念：

```text
acceptance probability
≠
Target 对这个 Token 的概率
```

它是在做一种“纠偏”。

Draft 从 `q` 分布提出候选，Target 希望最终仍服从 `p` 分布。接受 / 拒绝规则就是为了把 Draft 的提案校正回 Target。

## 4. 为什么拒绝后还需要 Correction Distribution？

如果某个 Draft Token 被拒绝，不能简单地：

```text
直接从 Target p 重新采样
```

因为我们已经利用过一次 Draft 的提案过程。

经典 speculative sampling 在拒绝位置使用修正分布：

\[
p'(x) \propto \max(0, p(x)-q(x))
\]

直觉是：

> Draft 已经过度覆盖的概率质量不应该再重复计算；剩余概率质量由 Target 补回来。

这一步和前面的接受概率配合，才保证最终分布不变。

In [ ]:
import numpy as np

p = np.array([0.50, 0.30, 0.20])  # target
q = np.array([0.70, 0.20, 0.10])  # draft

residual = np.maximum(0, p - q)
residual = residual / residual.sum()

print("Target p:", p)
print("Draft  q:", q)
print("拒绝后的 residual distribution:", np.round(residual, 3))

这个例子里 Draft 对第一个 Token 过于自信：

```text
q0 = 0.70
p0 = 0.50
```

所以 residual distribution 不会继续给它概率。

这就是为什么 speculative decoding 的“正确性”比“小模型猜、大模型验”复杂得多。

## 5. 实现一个最小 Speculative Sampling

我们不需要真的加载两个 LLM。

只要准备两个离散分布：

```text
Target p
Draft  q
```

然后实现：

1. 从 Draft 采一个 Token。
2. 按 `min(1, p/q)` 决定接受。
3. 拒绝时从 residual distribution 采样。

先只做一个位置。

In [ ]:
def speculative_one_step(p, q, rng):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()

    draft_token = rng.choice(len(q), p=q)
    accept_prob = min(1.0, p[draft_token] / q[draft_token])

    if rng.random() < accept_prob:
        return draft_token, True

    residual = np.maximum(0.0, p - q)
    if residual.sum() == 0:
        residual = p
    else:
        residual = residual / residual.sum()

    corrected = rng.choice(len(p), p=residual)
    return corrected, False

rng = np.random.default_rng(42)
p = [0.50, 0.30, 0.20]
q = [0.70, 0.20, 0.10]

for _ in range(8):
    token, accepted = speculative_one_step(p, q, rng)
    print("token =", token, "draft accepted =" , accepted)

## 6. 最关键的实验：分布真的没有变吗？

如果投机采样实现正确，跑很多次之后：

```text
Target-only sampling
```

和：

```text
Speculative sampling
```

得到的 Token 频率应该接近。

这比只统计“接受率”更重要，因为它验证了算法最核心的性质：

> **加速，但不改变 Target 原本的采样分布。**

In [ ]:
trials = 100_000
rng_target = np.random.default_rng(0)
rng_spec = np.random.default_rng(1)

target_samples = rng_target.choice(3, size=trials, p=np.array(p)/sum(p))
target_freq = np.bincount(target_samples, minlength=3) / trials

spec_samples = []
accepted_count = 0
for _ in range(trials):
    token, accepted = speculative_one_step(p, q, rng_spec)
    spec_samples.append(token)
    accepted_count += int(accepted)

spec_freq = np.bincount(spec_samples, minlength=3) / trials

print("token  target-only  speculative")
for i in range(3):
    print(f"{i:>5d}  {target_freq[i]:>10.4f}  {spec_freq[i]:>11.4f}")
print("\nDraft acceptance rate:", accepted_count / trials)

如果两列频率接近，就说明我们没有因为“先用小模型猜”而改变 Target Distribution。

以后看到技术报告说：

```text
lossless speculative decoding
distribution-preserving
exact sampling
```

这里的 “lossless / exact” 通常不是说：

> 输出字符串和普通采样每次完全一样。

而是说：

> **随机变量的目标分布不变。**

## 7. Acceptance Rate 为什么决定收益？

每轮 Draft 猜 `K` 个 Token。

如果平均只接受 0.5 个，收益很差。

如果平均接受 3~4 个，就可能一次昂贵 Target Forward 产出多个 Token。

一个简单指标：

```text
accepted tokens per target step
```

越高越好。

但 K 也不能无限增加：

```text
K 太小 → 没吃满一次 Target 验证
K 太大 → Draft 白猜很多，Target 验证也更重
```

所以最优 K 和模型对、工作负载、硬件都有关。

In [ ]:
def rough_speedup(k, accept_rate, draft_ms=5, verify_ms=45, target_step_ms=40):
    expected_tokens = 1 + k * accept_rate
    speculative_time = k * draft_ms + verify_ms
    baseline_time = expected_tokens * target_step_ms
    return baseline_time / speculative_time

for k in [2, 4, 6, 8]:
    for ar in [0.3, 0.6, 0.9]:
        print(f"K={k}, accept={ar:.1f} -> rough speedup={rough_speedup(k, ar):.2f}x")
    print()

这个公式只是 toy model，但它让你知道厂商报告里的 speedup 为什么一定要同时看：

- Draft / Target 模型是什么？
- Draft length / speculation length 是多少？
- Acceptance rate 多高？
- Batch size / concurrency 多大？
- 是 TPOT 提升还是吞吐提升？

单独写一句“Speculative Decoding 2x”信息远远不够。

## 8. Draft Model 不一定真的是“小模型”

最经典形式是：

```text
Small Draft Model
→ Large Target Model
```

但今天“投机”已经发展出很多变体：

### Independent Draft Model
一个独立小模型负责 Draft。

### Self-Speculative Decoding
Target 自己的浅层 / early exit / 子网络做 Draft。

### Medusa / EAGLE 一类路线
额外预测头或特定 Draft 结构一次提出多个候选。

### Prompt Lookup / N-gram Speculation
如果上下文里已经出现相同模式，直接从 Prompt 候选中猜。

所以招聘 JD 或厂商报告里看到：

```text
speculative decoding
speculative sampling
draft model
Medusa
EAGLE
multi-token prediction
```

不要先死记算法名。

先问：

> **候选是谁提出的？Target 怎么验证？一次能接受多少 Token？**

## 9. Speculative Decoding 在系统里放在哪里？

回到整个 Decode Stack：

```text
Target Model
   ↓
一次 Forward
   ↓
原来：确认 1 Token

Speculative:
Draft 提 K 个
   ↓
Target 一次 Verify
   ↓
确认 1~K+1 个 Token
```

它和量化不是同一类优化：

```text
Quantization
→ 每一次 Forward 更便宜

Speculative Decoding
→ 每一次昂贵 Forward 尽量产出更多 Token
```

两者甚至可以叠加。

## 10. 下一步：单请求快了，100 个请求一起进来怎么办？

到现在为止，我们一直站在**单个请求**视角：

- 20：Logits 怎么变成 Token？
- 21：一次 Token 为什么慢？
- 22：怎么让一次 Forward 更便宜？
- 23：怎么让一次 Target Forward 确认更多 Token？

但生产服务里不是一个用户。

可能同时有：

```text
请求 A: prompt=20, output=30
请求 B: prompt=20000, output=100
请求 C: prompt=1000, output=2000
...
```

下一章开始切换视角：

> **当很多请求共享一张 GPU，系统应该怎么排队、Batch、分 KV Cache、处理长 Prompt？**

这会进入 **Continuous Batching、PagedAttention、Prefix Caching、RadixAttention、Chunked Prefill、PD 分离**。

## 小结

- **Draft Model**：便宜地提出多个候选。
- **Target Model**：并行验证候选。
- **Acceptance Probability**：`min(1, p/q)`。
- **Correction Distribution**：拒绝后用 residual probability 修正。
- **Distribution-preserving**：最终采样分布与 Target-only 一致。
- **Acceptance Rate**：直接影响每次 Target Forward 能赚到多少 Token。
- **Speculation Length K**：太小吃不满，太大浪费 Draft。
- **Medusa / EAGLE / Self-Speculative**：都可以先按“谁 Draft、怎么 Verify”来理解。

## 作业

1. 手算 `p=[0.5,0.3,0.2]`、`q=[0.7,0.2,0.1]` 时三个 Token 的接受概率。
2. 为什么拒绝后不能简单“重新从 p 采样”？
3. 修改 Monte Carlo 实验，换三组不同 q，观察 Draft 越接近 p 时 Acceptance Rate 怎么变化。
4. 解释“lossless speculative decoding”里的 lossless 通常指什么。